In [1]:
from client import *
from common_imports import *

In [3]:
 # Agent instructions
summarizer_instructions="""
Summarize the customer's feedback in one short sentence. Keep it neutral and concise.
Example output:
App crashes during photo upload.
User praises dark mode feature.
"""

classifier_instructions="""
Classify the feedback as one of the following: Positive, Negative, or Feature request.
"""

action_instructions="""
Based on the summary and classification, suggest the next action in one short sentence.
Example output:
Escalate as a high-priority bug for the mobile team.
Log as positive feedback to share with design and marketing.
Log as enhancement request for product backlog.
"""

In [4]:
# Create the chat client


    # Create agents


    # Initialize the current feedback


    # Build sequential orchestration


    # Run and collect outputs


    # Display outputs
    

In [5]:
import os
from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient
from azure.identity.aio import DefaultAzureCredential

chat_client=FoundryChatClient(
        project_endpoint=MICROSOFT_FOUNDRY,
        model=AZURE_OPENAI_DEPLOYMENT_ID,
        credential=DefaultAzureCredential()
    )

In [6]:
# Create agents
summarizer_agent = chat_client.as_agent(
    name="summarizer",
    instructions=summarizer_instructions,
)

classifier_agent = chat_client.as_agent(
    name="classifier",
    instructions=classifier_instructions,
)

action_agent = chat_client.as_agent(
    name="action",
    instructions=action_instructions,
)

In [7]:
# Initialize the current feedback
feedback="""
I use the dashboard every day to monitor metrics, and it works well overall. 
But when I'm working late at night, the bright screen is really harsh on my eyes. 
If you added a dark mode option, it would make the experience much more comfortable.
"""

In [8]:
# Build sequential orchestration
workflow = SequentialBuilder(
    participants=[summarizer_agent, classifier_agent, action_agent],
    output_from="all",
).build()

In [9]:
# Run and collect outputs
result = await workflow.run(f"Customer feedback: {feedback}")
outputs = result.get_outputs()

In [11]:
# Display outputs
i = 1
for response in outputs:
    for msg in cast(list[Message], response.messages):
        name = msg.author_name or ("assistant" if msg.role == "assistant" else "user")
        print(f"{'-' * 60}\n{i:02d} [{name}]\n{msg.text}")
        i += 1

------------------------------------------------------------
01 
User suggests adding a dark mode for a more comfortable experience at night.

------------------------------------------------------------
02 
Feature request

------------------------------------------------------------
03 
Log as an enhancement request for the product backlog.